# LSTM Next Word Predictor (Beginner Friendly)
Welcome! In this project, you will build a Deep Learning model that can predict the next word in a sentence. This is the exact technology used behind smart keyboards, auto-correct systems, and text generation algorithms.

### What is an LSTM?
A **Long Short-Term Memory (LSTM)** network is a special type of Recurrent Neural Network (RNN) designed to learn long-term dependencies in sequential data (like text). 

Standard neural networks look at a single input and guess an output. But language is context-dependent. The word **"mat"** in *"The cat sat on the..."* is predicted because of the words that came *before* it. LSTMs remember historical information through special mechanisms called **gates**:
1. **Forget Gate**: Decides what information from previous words is no longer useful and should be thrown away.
2. **Input Gate**: Decides what new information from the current word should be added to the cell's memory.
3. **Output Gate**: Decides what the next hidden state (the output) should be, which is used to make the actual prediction.

Let's build this step-by-step in Keras/TensorFlow!

## Step 1: Import Necessary Libraries
We need **TensorFlow** for building the deep learning model, **NumPy** for numerical calculations, and **Matplotlib** to plot our training progress.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## Step 2: Define a Sample Dataset
For this Colab tutorial, we will use a small, self-contained text corpus. In real-world applications, you could load entire books (like Harry Potter or Shakespeare) as your training dataset. 

Let's define a short list of inspirational sentences about Artificial Intelligence and Deep Learning.

In [ ]:
# A simple text dataset
data = """
Deep learning is a subset of machine learning that is based on artificial neural networks.
Recurrent neural networks are designed to analyze sequential data like text or time series.
An LSTM or Long Short-Term Memory network is a special type of RNN capable of learning long term dependencies.
Next word prediction is a fascinating NLP task that powers systems like auto correct and predictive search engines.
We will build a simple next word predictor using Keras and TensorFlow.
With enough training data, the model can generate creative sentences word by word.
Keep learning and experimenting with deep learning projects.
"""

# Clean the text: convert to lowercase and split it line by line
corpus = data.lower().split("\n")
corpus = [line.strip() for line in corpus if line.strip()]

print("Number of lines in dataset:", len(corpus))
for index, line in enumerate(corpus):
    print(f"Line {index+1}: {line}")

## Step 3: Tokenization (Converting Words to Numbers)
Computers do not understand words directly; they understand numbers. We use a **Tokenizer** to map every unique word in our corpus to a unique integer index.

For example:
- `"deep"` -> `1`
- `"learning"` -> `2`
- `"is"` -> `3`

Then, we convert each sentence into a sequence of numbers.

In [ ]:
# Initialize and fit the tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)

# The vocabulary size is the number of unique words + 1 (reserved index 0 for padding)
total_words = len(tokenizer.word_index) + 1

print("Total unique words (Vocab Size):", total_words)
print("Word Mapping Dictionary:")
print(list(tokenizer.word_index.items())[:15], "...")

## Step 4: Generating Training Sequences
To train our next-word predictor, we need input features ($X$) and target labels ($y$).

Let's take the sentence: **"deep learning is fun"**
We split this into incremental sequences:
1. Input: `"deep"` | Target: `"learning"`
2. Input: `"deep learning"` | Target: `"is"`
3. Input: `"deep learning is"` | Target: `"fun"`

We then convert these words to their index sequences and **pad** them at the beginning (`pre`) so they all have the same length. This makes training computationally efficient.

In [ ]:
input_sequences = []
for line in corpus:
    # Convert words to numbers for this line
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    # Create N-grams (incremental sub-sequences)
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print(f"Total training sequences generated: {len(input_sequences)}")
print("Example sequence before padding:", input_sequences[0])

# Find the length of the longest sentence
max_sequence_len = max([len(x) for x in input_sequences])
print("Maximum sequence length:", max_sequence_len)

# Pad the sequences so that all sequences have the same length (max_sequence_len)
input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))
print("Example sequence after padding:\n", input_sequences[0])

## Step 5: Splitting Features (X) and Labels (y)
- **X (Features)**: Every token in the padded sequence *except* the last one.
- **y (Labels)**: The last token in the sequence (the word we want to predict). We convert this to a **one-hot encoded vector** so the model can calculate categorical cross-entropy loss.

In [ ]:
# X is all tokens except the last one
X = input_sequences[:, :-1]
# y is the last token (target)
y = input_sequences[:, -1]

# One-hot encode the labels
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print("X shape (samples, input_length):", X.shape)
print("y shape (samples, vocabulary_size):", y.shape)
print("First sample input (X[0]):", X[0])
print("First sample output index (y[0]):", np.argmax(y[0]))

## Step 6: Building the LSTM Model

Our neural network architecture contains:
1. **Embedding Layer**: Converts a word's integer index into a dense vector of a fixed size (e.g., 64 dimensions). This helps the model group words with similar meanings together (like "cat" and "dog").
2. **LSTM Layer**: Processes the sequence of embeddings. The 100 units act as the "memory cells" that learn the temporal patterns of language.
3. **Dropout Layer**: Randomly turns off some neural connections (20%) during training to prevent the model from memorizing the text too strictly (overfitting).
4. **Dense Layer (with Softmax)**: Outports a probability distribution across all possible words in our vocabulary. Softmax outputs sum up to 1.0 (e.g., "cat" has 0.8 probability, "dog" has 0.1, etc.).

In [ ]:
model = Sequential()

# Embedding Layer: Input length is (max_sequence_len - 1) because the last token was removed for y
model.add(Embedding(input_dim=total_words, output_dim=64, input_length=max_sequence_len-1))

# LSTM Layer: Learn sequential patterns
model.add(LSTM(100, return_sequences=False))

# Dropout: Regularize to prevent overfitting
model.add(Dropout(0.2))

# Dense Output Layer: Predict probability of each word in vocabulary
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Print model summary structure
model.summary()

## Step 7: Train the Model
Now we train our model. Since this is a very small dataset, we'll train it for 150 epochs so the model learns it perfectly. This will take only a few seconds!

In [ ]:
# Train the model
epochs = 150
history = model.fit(X, y, epochs=epochs, batch_size=16, verbose=1)

print("Training Completed!")

## Step 8: Plot Training Metrics
It is always a good practice to visualize your training performance to ensure the loss is dropping and accuracy is rising.

In [ ]:
# Extract metrics
acc = history.history['accuracy']
loss = history.history['loss']
epochs_range = range(1, epochs + 1)

# Plot figures
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Accuracy', color='blue')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Loss', color='red')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

## Step 9: Make Predictions!
To predict the next word, we take a seed text (e.g. `"deep learning"`), tokenize it, pad it to the model's expected shape, feed it to the model, and convert the highest probability index back to the corresponding word.

In [ ]:
def generate_next_words(seed_text, next_words_limit=3):
    generated_sentence = seed_text
    
    for _ in range(next_words_limit):
        # Convert the current text to word indices
        token_list = tokenizer.texts_to_sequences([generated_sentence])[0]
        
        # Pad the sequence so it matches the expected input length
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        
        # Run the model to get probability distribution
        predicted_probabilities = model.predict(token_list, verbose=0)
        
        # Find the index with the highest probability
        predicted_index = np.argmax(predicted_probabilities, axis=-1)[0]
        
        # Find the word mapped to this index
        predicted_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                predicted_word = word
                break
        
        # If we didn't find a word, stop generating
        if not predicted_word:
            break
            
        # Add the word to the output sentence
        generated_sentence += " " + predicted_word
        
    return generated_sentence

# Test cases
print(generate_next_words("deep learning is", next_words_limit=2))
print(generate_next_words("recurrent neural", next_words_limit=2))
print(generate_next_words("we will build", next_words_limit=4))

## Conclusion & What's Next?
Congratulations! You just built an LSTM model that can read sequences and predict subsequent words.

**Try editing the notebook**:
1. Replace the `data` variable content with your own text files (such as a news article or short book chapter) and train it.
2. Try altering model hyperparameters like changing the Embedding Output Dimension (e.g. from 64 to 128) or adding another LSTM layer: `model.add(LSTM(100, return_sequences=True))` followed by another `model.add(LSTM(50))`. Observe how accuracy and training speed change!